In [1]:
!pip install langchain_groq

In [2]:
!pip install langchain_core

In [3]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Creating model instance
llama_model = ChatGroq(
    model = "llama-3.3-70b-versatile",
    groq_api_key = "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",
    max_tokens = 128,
    temperature = 0.5,
    
)
message = llama_model.invoke([
    SystemMessage(content= "You are a helpful AI that assists the user in choosing the perfect book in one sentence"),
    HumanMessage(content = "What horror Book should i read?")
])
print(f" the message type is :{type(message)}")
print(f" The response of ai is {message.content}")

 the message type is :<class 'langchain_core.messages.ai.AIMessage'>
 The response of ai is If you're looking for a chilling horror book, I highly recommend "The Shining" by Stephen King, a classic tale of isolation and supernatural terror that will keep you on the edge of your seat.


In [5]:
msg = llama_model.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)
print(f" The response is {msg.content}")

 The response is Aim to attend 3-4 CrossFit classes per week for optimal results and progressive overload.


In [6]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder,PromptTemplate
prompt = ChatPromptTemplate.from_messages([
 ("system", "You are a helpful assistant"),
 ("user", "Tell me a joke about {topic}")
])

# Create a dictionary with the variable to be inserted into the template
# The key "topic" matches the placeholder name in the user message
input_ = {"topic": "cats"}

# Format the chat template with our input values
# This replaces {topic} with "cats" in the user message
# The result will be a formatted chat message structure ready to be sent to a model
prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cats', additional_kwargs={}, response_metadata={})])

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system"," You are a helpful ai assistant"),
    MessagesPlaceholder("msgs")
])
input_ ={"msgs": [HumanMessage(content = "What day is after Tuesday?")]}
prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content=' You are a helpful ai assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What day is after Tuesday?', additional_kwargs={}, response_metadata={})])

In [8]:
chain = prompt | llama_model
response = chain.invoke(input_)
print(response)

content='The day after Tuesday is Wednesday.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 47, 'total_tokens': 55, 'completion_time': 0.039814311, 'completion_tokens_details': None, 'prompt_time': 0.002482565, 'prompt_tokens_details': None, 'queue_time': 0.063813249, 'total_time': 0.042296876}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019bfe66-5d43-7033-b8d9-574fa45e5471-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 8, 'total_tokens': 55}


In [9]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [10]:
class Joke (BaseModel):
    setup : str = Field(description = "Question to setup a joke")
    punchline: str = Field(description = "Answer to resolve the Joke")
    

In [11]:
joke_query = "Tell me a joke"
output_parser = JsonOutputParser(pydantic_object = Joke)
format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template = "Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables = ["query"],
    partial_variables = {"format_instructions":format_instructions}
)
chain = prompt | llama_model | output_parser
response = chain.invoke({"query":joke_query})
print(f"{response}")

{'setup': "Why couldn't the bicycle stand up by itself?", 'punchline': 'Because it was two-tired.'}


In [12]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template = "Answer the user query.\n{format_instructions}\nList 5 {subject}\n",
    input_variables = ["subject"],
    partial_variables = {"format_instructions":format_instructions}
)
chain = prompt | llama_model | output_parser
response = chain.invoke({"subject":"fruits"})
print(f"{response}")

['apple', 'banana', 'mango', 'orange', 'grape']


In [13]:
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)
movie_chain = prompt_template | llama_model | json_parser
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name":movie_name})
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: The Wachowskis
Year: 1999
Genre: Science Fiction


In [45]:
# Import the Document class from langchain_core.documents module
# Document is a container for text content with associated metadata
from langchain_core.documents import Document

# Create a Document instance with:
# 1. page_content: The actual text content about Python
# 2. metadata: A dictionary containing additional information about this document
Document(page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
metadata={
    'my_document_id' : 234234,                      # Unique identifier for this document
    'my_document_source' : "About Python",          # Source or title information
    'my_document_create_time' : 1680013019          # Unix timestamp for document creation (March 28, 2023)
 })

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language.\n Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

In [14]:

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()
document[2]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 2, 'page_label': '3'}, page_content='Figure 2. An AIMessage illustration \nC. Prompt Template \nPrompt templates [10] allow you to structure input for LLMs. \nThey provide a convenient way to format user inputs and \nprovide instructions to generate responses. Prompt templates \nhelp ensure that the LLM understands the desired context and \nproduces relevant outputs. \nThe prompt template classes in LangChain are built to \nmake constructing prompts with dynamic inputs easier. Of \nthese classes, the simplest is the PromptTemplate. \nD. Chain \nChains [11] in LangChain refer to the combination of \nmultiple components to achieve specific tasks. Th

In [15]:
 #Import the WebBaseLoader class from langchain_community's document_loaders module
# This loader is designed to scrape and extract text content from web pages
from langchain_community.document_loaders import WebBaseLoader

# Create a WebBaseLoader instance by passing the URL of the web page to load
# This URL points to the LangChain documentation's introduction page
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

# Call the load() method to:
# 1. Send an HTTP request to the specified URL
# 2. Download the HTML content
# 3. Parse the HTML to extract meaningful text
# 4. Create a list of Document objects containing the extracted content
web_data = loader.load()

# Print the first 1000 characters of the page content from the first Document
# This provides a preview of the successfully loaded web content
# web_data[0] accesses the first Document in the list
# .page_content accesses the text content of that Document
# [:1000] slices the string to get only the first 1000 characters
print(web_data[0].page_content[:1000])

USER_AGENT environment variable not set, consider setting it to identify your requests.


LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageLangChain + LangGraphSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewLangChainLangGraphDeep AgentsIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewBuilt-in middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolvesCopy pageLangChain is the easiest way to start building agents and ap

In [16]:

from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size = 200, chunk_overlap = 20, separator = "\n",strip_whitespace = True)
chunks = text_splitter.split_documents(document)
print(len(chunks))

147


In [17]:
chunks[5].page_content

'individuals seeking guidance and support in these critical areas. \nMindGuide lever ages the capabilities of LangChain and its \nChatModels, specifically Chat OpenAI, as the bedrock of its'

In [18]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Load the LangChain paper
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document)
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
display_document_stats(chunks_2, "Splitter 2")



=== Splitter 1 Statistics ===
Total number of chunks: 95
Average chunk size: 263.80 characters
Metadata keys preserved: creationdate, source, total_pages, page, page_label, creator, moddate, title, author, producer

Example chunk:
Content (first 150 chars): comprehensive support within the field of mental health. 
Additionally, the paper discusses the implementation of 
Streamlit to enhance the user ex pe...
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Min chunk size: 49 characters
Max chunk size: 299 characters

=== Splitter 2 Statistics ===
Total number of chunks: 57
Average chunk size: 452.74 characters
Metadata keys preserved: creationdate, source, total_pages, page, page_

In [19]:
!pip install langchain-huggingface sentence-transformers

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
vectors = embeddings.embed_documents([chunk.page_content for chunk in chunks])
print(f"Number of chunks: {len(vectors)}")
print(f"Vector dimension: {len(vectors[0])}")
print(f"First 5 values of first chunk: {vectors[0][:5]}")

Number of chunks: 147
Vector dimension: 384
First 5 values of first chunk: [0.000882672262378037, 0.013385623693466187, 0.020798644050955772, -0.008458723314106464, -0.06765628606081009]


In [67]:
!pip install langchain.vectorstores

ERROR: Could not find a version that satisfies the requirement langchain.vectorstores (from versions: none)
ERROR: No matching distribution found for langchain.vectorstores


In [9]:
pip install chromadb


Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip install docarray


  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [docarray]4/5 [docarray]
II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models


In [21]:
from langchain_community.vectorstores import Chroma,DocArrayInMemorySearch
docsearch = DocArrayInMemorySearch.from_documents(chunks , embeddings)
docs = docsearch.similarity_search("Langchain")
print(docs[0].page_content)

II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models


In [22]:
retriever = docsearch.as_retriever()
docs = retriever.invoke("Langchain")
print(docs[0])

page_content='II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models' metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


In [16]:
!pip install langchain

In [28]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function to format retrieved documents
def format_docs(docs):
    """Concatenate all retrieved documents into a single string"""
    return "\n\n".join([doc.page_content for doc in docs])

# Prompt template
template = """Answer the question based only on the following context:

{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Create the RAG chain (equivalent to RetrievalQA)
qa_chain = (
    {
        "context": docsearch.as_retriever() | format_docs,  # Retrieve and format docs
        "question": RunnablePassthrough()  # Pass question through
    }
    | prompt  # Format the prompt
    | llama_model  # Send to LLM
    | StrOutputParser()  # Parse output to string
)

# Define a query
query = "what is this paper discussing?"

# Execute the chain
# This will:
# 1. Send the query to the retriever to get relevant documents
# 2. Combine those documents using format_docs (like "stuff" method)
# 3. Send the query and combined documents to the Llama LLM
# 4. Return the generated answer
result = qa_chain.invoke(query)

print(result)

This paper appears to be discussing mental health, specifically the complexities of addressing mental health challenges and the role of a framework called LangChain in facilitating interactions and support for individuals struggling with their mental health.


In [18]:
import langchain
print(langchain.__version__)

1.2.7


In [30]:
!pip install langchain.memory

ERROR: Could not find a version that satisfies the requirement langchain.memory (from versions: none)
ERROR: No matching distribution found for langchain.memory


In [32]:
from langchain_community.chat_message_histories import ChatMessageHistory
chat = llama_model
history = ChatMessageHistory()
history.add_ai_message("Hi")
history.add_user_message("What is the capital of USA?.")


In [33]:
history.messages

[AIMessage(content='Hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the capital of USA?.', additional_kwargs={}, response_metadata={})]